In [1]:
import bempp_cl.api
import numpy as np
import pytest


In [2]:
# Physical parameters
B_0 = 1.0  # Incident field magnitude

mu_0 = 1.0  # Normalized permeability

# Create geometry (sphere)
grid = bempp_cl.api.shapes.sphere(h=0.2, r=1.0)
print(f"Grid: {grid.number_of_elements} elements")




Grid: 846 elements


In [3]:
# grid.plot()

In [4]:
div_space  = bempp_cl.api.function_space(grid, "RWG", 0)
curl_space = bempp_cl.api.function_space(grid, "SNC", 0)
p1_space   = bempp_cl.api.function_space(grid, "P", 1) 
print(f"Div space dimension: {div_space.global_dof_count}")
print(f"Curl space dimension: {curl_space.global_dof_count}")
print(f"P1 space dimension: {p1_space.global_dof_count}")


Div space dimension: 1269
Curl space dimension: 1269
P1 space dimension: 425


In [5]:
@bempp_cl.api.real_callable
def incident_field_tangential(x, n, domain_index, result):
    B_inc = np.array([0.0 * x[2], 0.0 * x[2], B_0 + 0.0 * x[2]])
    result[:] = np.cross(B_inc, n)


In [6]:
# Create grid function for RHS
rhs = bempp_cl.api.GridFunction(div_space, fun=incident_field_tangential, dual_space=curl_space)
print(f"RHS grid function created")



RHS grid function created


In [7]:
# Single layer boundary operator
V_op = bempp_cl.api.operators.boundary.maxwell.single_layer(div_space, div_space, curl_space)

# Single layer boundary operator
K_op = bempp_cl.api.operators.boundary.maxwell.double_layer(div_space, div_space, curl_space)


In [8]:
# Maxwell identity operator (for proper inner product)
identity_p1 = bempp_cl.api.operators.boundary.sparse.identity(
    p1_space, p1_space, p1_space
)


identity = bempp_cl.api.operators.boundary.sparse.identity(
    div_space, div_space, curl_space
)


vector_grad = bempp_cl.api.operators.boundary.sparse._vector_grad_product(
    p1_space, div_space, curl_space
)


In [9]:
identity_p1.weak_form()

<425x425 SparseDiscreteBoundaryOperator with dtype=float64>

In [10]:
# Solve the linear system
# from bempp_cl.api.linalg import lu, gmres

# Solution
# j_solution = lu(V_op, rhs)

In [11]:
# help(gmres)

In [12]:
from bempp_cl.api.linalg import lu, gmres

j_solution, j_info, j_count = gmres(V_op, rhs, tol=1e-6, return_iteration_count=True)

print(f"Number of GMRES iterations: {j_count}")

Number of GMRES iterations: 45


In [13]:
jmin = np.min(j_solution.coefficients)
jmax = np.max(j_solution.coefficients)

print(f"Minimum value: {jmin}\nMaximum value: {jmax}")

Minimum value: -1.374719751547766
Maximum value: 1.3963172193141946


## Block operator

We assemble the block operator
\begin{equation*}
A_h = \begin{pmatrix} 
V_h & B^T_h \\ 
B_h^T & 0
\end{pmatrix},
\end{equation*}
where

- $V_h$ is the Galerkin boundary element matrix associated to the (vector) single layer operator.
- $B_h^T$ is the Galerkin matrix for the gradient of P1 functions, tested with RT.
- $B_h$ is the weak form of the surface divergence operator.

Matrix $A_h$ then corresponds to a discretization of the (vector) single layer operator and a Lagrange multiplier for the divergence-free condition on the fields.
  


In [14]:
# Block operator
from bempp_cl.api.assembly.blocked_operator import BlockedDiscreteOperator


blocks = [[None, None], [None, None]]

B = vector_grad.weak_form()

blocks[0][0] = V_op.weak_form()
blocks[0][1] = B
blocks[1][0] = -B.T
blocks[1][1] = 0*identity_p1.weak_form()


A = BlockedDiscreteOperator(np.array(blocks))


In [18]:
rhs_p1 = np.zeros(p1_space.global_dof_count, dtype=float)

RHS = np.concatenate([rhs.projections(curl_space), rhs_p1])

In [19]:
RHS

array([ 0.01331187, -0.01843508,  0.00875821, ...,  0.        ,
        0.        ,  0.        ], shape=(1694,))

In [20]:

from scipy.sparse.linalg import gmres

it_count = 0

def count_iterations(x):
    global it_count
    it_count += 1

SOL, info = gmres(A, RHS, callback=count_iterations)

print(f"Number of GMRES iterations: {it_count}")
print(f"GMRES converged to a solution: {info == 0}")

Number of GMRES iterations: 11077
GMRES converged to a solution: True


In [21]:
info

0

In [22]:
sol_bem    = SOL[:div_space.global_dof_count]
sol_lambda = SOL[div_space.global_dof_count:]

In [23]:
sol_bem

array([ 0.75193245, -1.23180282,  0.54000454, ..., -0.2935851 ,
       -1.01995095,  0.16113853], shape=(1269,))

In [24]:
sol_lambda

array([ 1.85154889e-05, -6.55446009e-07, -1.73665328e-05,  8.86344999e-06,
        1.04615411e-06,  3.83619073e-07,  1.69256946e-05,  1.07826810e-05,
        2.63876529e-06,  2.59163088e-06,  1.80140127e-06, -1.48688886e-05,
       -1.02478521e-05,  5.95956237e-06,  1.18750847e-05, -1.58262448e-05,
       -1.65228524e-05, -1.43498984e-05, -2.02996900e-05, -2.38172750e-05,
       -8.63560083e-06, -8.17968945e-06, -1.25773328e-05, -4.98561531e-07,
       -8.80061122e-06, -1.42600119e-05, -4.08821791e-06,  1.02296045e-05,
        2.14656914e-05,  2.30313906e-05,  1.85480919e-05,  1.47505820e-05,
       -6.20775648e-06, -8.20031675e-06, -2.85928781e-06, -2.14508870e-05,
       -1.73066167e-05, -2.83125175e-05,  3.09265071e-05,  8.40415506e-05,
        1.29553252e-05, -4.75463209e-05, -2.22418292e-05, -1.64903315e-05,
        1.18149642e-05,  1.38063122e-05,  3.03008298e-05,  1.52689066e-05,
        3.80176075e-06,  1.38524166e-05,  3.04494891e-05,  1.88625569e-05,
        9.92963263e-06, -

In [25]:
# slp_op.weak_form().to_dense()